In [20]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.transformed('train_t02.parquet'))
test  = pd.read_parquet(DataLoader.transformed('test_t02.parquet'))
print('Train shape BEFORE pruning:', train.shape)
print('Test  shape BEFORE pruning:', test.shape)
print()
print('Columns:', list(train.columns))

Train shape BEFORE pruning: (137176, 20)
Test  shape BEFORE pruning: (34294, 20)

Columns: ['Inspection ID', 'License #', 'Facility Type', 'Risk', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION', 'APPLICATION TYPE', 'LICENSE STATUS', 'violations_recorded', 'license_matched', 'has_prior_inspection', 'days_to_license_expiry', 'license_expiry_missing']


In [21]:
train = train.sort_values(['License #', 'Inspection Date']).reset_index(drop=True)
test  = test.sort_values(['License #', 'Inspection Date']).reset_index(drop=True)

In [22]:
train['days_since_last_inspection'] = (
    train.groupby('License #')['Inspection Date']
    .diff().dt.days
)
test['days_since_last_inspection'] = (
    test.groupby('License #')['Inspection Date']
    .diff().dt.days
)

print(train['Inspection Date'].isna().sum(), 'missing values in train Inspection Date')
print(train['days_since_last_inspection'].isna().sum(), 'missing values in train')
print(test['days_since_last_inspection'].isna().sum(), 'missing values in test')


0 missing values in train Inspection Date
28166 missing values in train
14272 missing values in test


In [23]:
# First inspection per establishment has no prior — fill with train median/mean
median_gap = train['days_since_last_inspection'].median()
train['days_since_last_inspection'] = train['days_since_last_inspection'].fillna(median_gap)
test['days_since_last_inspection']  = test['days_since_last_inspection'].fillna(median_gap)

print(f'Median gap: {median_gap:.0f} days')
print(f'Nulls — train: {train["days_since_last_inspection"].isna().sum()}, test: {test["days_since_last_inspection"].isna().sum()}')

Median gap: 203 days
Nulls — train: 0, test: 0


In [24]:
train['prev_inspection_result'] = (
    train.groupby('License #')['Results'].shift(1)
)
test['prev_inspection_result'] = (
    test.groupby('License #')['Results'].shift(1)
)

def fail_rate_last_3(series):
    return series.shift(1).rolling(window=3, min_periods=1).mean()

train['fail_rate_last_3'] = train.groupby('License #')['Results'].transform(fail_rate_last_3)
test['fail_rate_last_3']  = test.groupby('License #')['Results'].transform(fail_rate_last_3)


In [25]:
# First inspection per establishment — fill with train means
train['prev_inspection_result'] = train['prev_inspection_result'].fillna(train['prev_inspection_result'].mean())
test['prev_inspection_result']  = test['prev_inspection_result'].fillna(train['prev_inspection_result'].mean())

train['fail_rate_last_3'] = train['fail_rate_last_3'].fillna(train['fail_rate_last_3'].mean())
test['fail_rate_last_3']  = test['fail_rate_last_3'].fillna(train['fail_rate_last_3'].mean())

print(f'Nulls prev_inspection_result — train: {train["prev_inspection_result"].isna().sum()}, test: {test["prev_inspection_result"].isna().sum()}')
print(f'Nulls fail_rate_last_3       — train: {train["fail_rate_last_3"].isna().sum()}, test: {test["fail_rate_last_3"].isna().sum()}')

Nulls prev_inspection_result — train: 0, test: 0
Nulls fail_rate_last_3       — train: 0, test: 0


In [26]:
print('Inspection Type value counts:')
print(train['Inspection Type'].value_counts())

insp_fail_rate = train.groupby('Inspection Type')['Results'].mean()
train['inspection_type_encoded'] = train['Inspection Type'].map(insp_fail_rate)
test['inspection_type_encoded']  = test['Inspection Type'].map(insp_fail_rate)

global_mean = train['Results'].mean()
train['inspection_type_encoded'] = train['inspection_type_encoded'].fillna(global_mean)
test['inspection_type_encoded'] = test['inspection_type_encoded'].fillna(global_mean)

print(f'\nNulls — train: {train["inspection_type_encoded"].isna().sum()}, test: {test["inspection_type_encoded"].isna().sum()}')

Inspection Type value counts:
Inspection Type
Canvass                              66377
License                              18678
Canvass Re-Inspection                15855
Complaint                            13598
License Re-Inspection                 7196
                                     ...  
fire complaint                           1
License consultation                     1
OWNER SUSPENDED OPERATION/LICENSE        1
CANVASS/SPECIAL EVENT                    1
Kids Cafe'                               1
Name: count, Length: 103, dtype: int64

Nulls — train: 0, test: 0


In [27]:
facility_fail_rate = train.groupby('Facility Type')['Results'].mean()
train['facility_type_encoded'] = train['Facility Type'].map(facility_fail_rate)
test['facility_type_encoded']  = test['Facility Type'].map(facility_fail_rate)

test['facility_type_encoded'] = test['facility_type_encoded'].fillna(global_mean)
train['facility_type_encoded'] = train['facility_type_encoded'].fillna(global_mean)

print(f'Nulls — train: {train["facility_type_encoded"].isna().sum()}, test: {test["facility_type_encoded"].isna().sum()}')

Nulls — train: 0, test: 0


In [28]:
train['is_revoked'] = (train['LICENSE STATUS'] == 'REV').astype(int)
test['is_revoked']  = (test['LICENSE STATUS'] == 'REV').astype(int)

In [29]:
APP_TYPE_MAP = {
    'ISSUE':  0,  # new license
    'RENEW':  1,  # renewal
    'C_LOC':  2,  # change of location
    'C_EXPA': 2,  # expansion
    'C_CAPA': 2,  # capacity change
    'C_SBA':  2,  # change of ownership
}

train['application_type_encoded'] = train['APPLICATION TYPE'].map(APP_TYPE_MAP)
test['application_type_encoded']  = test['APPLICATION TYPE'].map(APP_TYPE_MAP)

train['application_type_encoded'] = train['application_type_encoded'].fillna(1)  # default to RENEW (modal)
test['application_type_encoded']  = test['application_type_encoded'].fillna(1)

In [30]:
NO_LONGER_NEEDED = [
    'Inspection ID', 'License #', 'Zip', 'Longitude', 'Latitude',
    'Violations', 'license_matched', 'Inspection Type', 'Facility Type',
    'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION', 'LICENSE STATUS', 'APPLICATION TYPE'
]

train.drop(columns=NO_LONGER_NEEDED, inplace=True)
test.drop(columns=NO_LONGER_NEEDED, inplace=True)

print('\nRemaining columns:', train.columns.tolist())


Remaining columns: ['Risk', 'Inspection Date', 'Results', 'violations_recorded', 'has_prior_inspection', 'days_to_license_expiry', 'license_expiry_missing', 'days_since_last_inspection', 'prev_inspection_result', 'fail_rate_last_3', 'inspection_type_encoded', 'facility_type_encoded', 'is_revoked', 'application_type_encoded']


In [ ]:
train.to_parquet(DataLoader.transformed('train_t03.parquet'), index=False)
test.to_parquet(DataLoader.transformed('test_t03.parquet'), index=False)
print('Saved train_t03.parquet and test_t03.parquet')